# M5 Agentic AI — Market Research Team (Summer Sunglasses Campaign)


## 1. Introduction

In this lab, you act as the technical AI lead at a fashion brand preparing a summer sunglasses campaign.
You will design an automated creative pipeline that:

1. Scans online sources for emerging fashion trends.
2. Matches those trends to sunglasses in an internal catalog.
3. Designs a campaign visual (prompt + image generation).
4. Generates a short marketing quote based on the image + trends.
5. Packages everything into an executive-ready markdown report.

The goal is to experience how multiple agents, tools, and models can be orchestrated into a single workflow.


### 1.1 Lab Overview


You will:

- Import libraries and set up the environment.
- Define available tools and test them.
- Define a team of agents:
  - Market Research Agent
  - Graphic Designer Agent
  - Copywriter Agent
  - Packaging Agent
- Wrap the full workflow into `run_sunglasses_campaign_pipeline()` and run it end-to-end.


### 1.2 Learning outcome


By the end, you will be able to:

- Build multi-agent pipelines that coordinate planning, research, and creative generation.
- Ground agent outputs in external tools and structured internal data.
- Add reflection/packaging steps that enforce quality and produce executive-ready artifacts.


## 2. Setup: Import libraries and load environment


In [1]:
# =========================
# Imports
# =========================

# --- Standard library ---
import base64
import json
import os
import re
from datetime import datetime
from io import BytesIO

# --- Third-party ---
import requests
import openai
from PIL import Image
from dotenv import load_dotenv
from IPython.display import Markdown, display
import aisuite

# --- Local / project ---
import tools
import utils

# =========================
# Environment & Client
# =========================
load_dotenv()
client = aisuite.Client()


## 3. Available Tools


Agentic pipelines become effective when the model is given explicit capabilities beyond base reasoning.

In this lab, we use:
- `tools.tavily_search_tool(query)` to perform live web searches (trend discovery)
- `tools.product_catalog_tool()` to fetch the internal sunglasses catalog


In [2]:
# Try web search tool:
tools.tavily_search_tool("trends in sunglasses fashion")


[{'title': 'The Only 5 Sunglass Trends That Matter in 2025 | Who What Wear',
  'content': "# Jackie O Shades and Matrix Lenses: The Only Sunglasses That Matter in 2025 Look Like This. There are the retro frames inspired by Jackie O's style in the 1960s that will speak to the elegant dressers among us, then there are the sleek Matrix-esque styles that Khaite is co-signing, and the metal wire-rim glasses that celebs like Suki Waterhouse and Daisy Edgar-Jones are ensuring are the new default style. Like so many of the biggest trends of 2025, sunglasses are taking inspiration from the 1960s. A pair of metal wire-frame sunglasses like the Gucci style Daisy Edgar-Jones has been wearing all over NYC recently, or the Celine Triomphe frames favored by Suki Waterhouse. If the 1960s are defining some of the coolest sunglass styles right now, then so are the 1970s. Embellished Aviator-Style Tortoiseshell Recycled-Acetate and Gold-Tone Sunglasses. Just when I thought I'd seen the last of the wrapar

In [3]:
# Try internal catalog tool:
tools.product_catalog_tool()


[{'name': 'Aviator',
  'item_id': 'SG001',
  'description': 'Originally designed for pilots, these teardrop-shaped lenses with thin metal frames offer timeless appeal. The large lenses provide excellent coverage while the lightweight construction ensures comfort during long wear.',
  'quantity_in_stock': 23,
  'price': 103},
 {'name': 'Wayfarer',
  'item_id': 'SG002',
  'description': 'Featuring thick, angular frames that make a statement, these sunglasses combine retro charm with modern edge. The rectangular lenses and sturdy acetate construction create a confident look.',
  'quantity_in_stock': 6,
  'price': 92},
 {'name': 'Mystique',
  'item_id': 'SG003',
  'description': 'Inspired by 1950s glamour, these frames sweep upward at the outer corners to create an elegant, feminine silhouette. The subtle curves and often embellished temples add sophistication to any outfit.',
  'quantity_in_stock': 3,
  'price': 88},
 {'name': 'Sport',
  'item_id': 'SG004',
  'description': 'Designed for 

## 4. Agent Definitions — Building Your Team


### 4.1 Market Research Agent


The Market Research Agent:
1) Scans the web for current sunglasses fashion trends using `tavily_search_tool`.
2) Cross-checks signals against the internal catalog using `product_catalog_tool`.
3) Returns a concise brief: top trends + matching products + justification.


In [4]:
def market_research_agent(return_messages: bool = False):
    utils.log_agent_title_html("Market Research Agent", "🕵️‍♂️")

    prompt_ = f"""
You are a fashion market research agent tasked with preparing a trend analysis for a summer sunglasses campaign.

Your goal:
1. Explore current fashion trends related to sunglasses using web search.
2. Review the internal product catalog to identify items that align with those trends.
3. Recommend one or more products from the catalog that best match emerging trends.
4. If needed, today date is {datetime.now().strftime("%Y-%m-%d")}.

You can call the following tools:
- tavily_search_tool: to discover external web trends.
- product_catalog_tool: to inspect the internal sunglasses catalog.

Once your analysis is complete, summarize:
- The top 2–3 trends you found.
- The product(s) from the catalog that fit these trends.
- A justification of why they are a good fit for the summer campaign.
"""
    messages = [{"role": "user", "content": prompt_}]
    tools_ = tools.get_available_tools()

    while True:
        response = client.chat.completions.create(
            model="openai:o4-mini",
            messages=messages,
            tools=tools_,
            tool_choice="auto"
        )

        msg = response.choices[0].message

        if msg.content:
            utils.log_final_summary_html(msg.content)
            return (msg.content, messages) if return_messages else msg.content

        if msg.tool_calls:
            for tool_call in msg.tool_calls:
                utils.log_tool_call_html(tool_call.function.name, tool_call.function.arguments)
                result = tools.handle_tool_call(tool_call)
                utils.log_tool_result_html(result)

                messages.append(msg)
                messages.append(tools.create_tool_response_message(tool_call, result))
        else:
            utils.log_unexpected_html()
            return ("[⚠️ Unexpected: No tool_calls or content returned]", messages) if return_messages else "[⚠️ Unexpected: No tool_calls or content returned]"


In [5]:
# Run market research
market_research_result = market_research_agent()


### 4.2 Graphic Designer Agent


This agent translates trend insights into a visual concept:

1) Use a text model to generate:
   - an image generation prompt
   - a short caption
2) Send the prompt to DALL·E 3 to generate the image.
3) Save the generated image locally for reuse.


In [6]:
def graphic_designer_agent(trend_insights: str, caption_style: str = "short punchy", size: str = "1024x1024") -> dict:
    """
    Uses aisuite to generate a marketing prompt/caption and OpenAI (directly) to generate the image.

    Args:
        trend_insights (str): Trend summary from the researcher agent.
        caption_style (str): Optional style hint for caption.
        size (str): Image resolution (e.g., '1024x1024').

    Returns:
        dict: A dictionary with image_url, prompt, caption, and image_path.
    """

    utils.log_agent_title_html("Graphic Designer Agent", "🎨")

    # Step 1: Generate prompt and caption using aisuite
    system_message = (
        "You are a visual marketing assistant. Based on the input trend insights, "
        "write a creative and visual prompt for an AI image generation model, and also a short caption."
    )

    user_prompt = f"""
Trend insights:
{trend_insights}

Please output:
1. A vivid, descriptive prompt to guide image generation.
2. A marketing caption in style: {caption_style}.

Respond in this format:
{{"prompt": "...", "caption": "..."}}
"""

    chat_response = client.chat.completions.create(
        model="openai:o4-mini",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_prompt}
        ]
    )

    content = (chat_response.choices[0].message.content or "").strip()
    match = re.search(r"\{.*\}", content, re.DOTALL)
    parsed = json.loads(match.group(0)) if match else {"error": "No JSON returned", "raw": content}

    prompt = parsed.get("prompt", "")
    caption = parsed.get("caption", "")

    # Step 2: Generate image directly using openai-python
    openai_client = openai.OpenAI()
    image_response = openai_client.images.generate(
        model="dall-e-3",
        prompt=prompt,
        size=size,
        quality="standard",
        n=1,
        response_format="url"
    )

    image_url = image_response.data[0].url

    # Save image locally
    img_bytes = requests.get(image_url).content
    img = Image.open(BytesIO(img_bytes))

    filename = os.path.basename(image_url.split("?")[0])
    image_path = filename
    img.save(image_path)

    # Log summary with local image
    utils.log_final_summary_html(f"""
        <h3>Generated Image and Caption</h3>

        <p><strong>Image Path:</strong> <code>{image_path}</code></p>

        <p><strong>Generated Image:</strong></p>
        <img src="{image_path}" alt="Generated Image" style="max-width: 100%; height: auto; border: 1px solid #ccc; border-radius: 8px; margin-top: 10px; margin-bottom: 10px;">

        <p><strong>Prompt:</strong> {prompt}</p>
    """)

    return {
        "image_url": image_url,
        "prompt": prompt,
        "caption": caption,
        "image_path": image_path
    }


In [7]:
# Generate campaign visual
graphic_designer_agent_result = graphic_designer_agent(
    trend_insights=market_research_result,
)


### 4.3 Copywriter Agent


This agent takes:
- the generated campaign image (local file)
- the trend summary

It sends multimodal input (image + text) to produce:
- a short, elegant campaign quote (<= 12 words)
- a justification tying the quote back to the image + trends


In [9]:
def copywriter_agent(image_path: str, trend_summary: str, model: str = "openai:o4-mini") -> dict:

    """
    Uses aisuite (OpenAI only) to send an image and a trend summary and return a campaign quote.

    Args:
        image_path (str): URL of the image to be analyzed.
        trend_summary (str): Text from the researcher agent.
        model (str): OpenAI model (e.g., openai:o4-mini, openai:gpt-4o)

    Returns:
        dict: {
            "quote": "...",
            "justification": "...",
            "image_path": "..."
        }
    """

    utils.log_agent_title_html("Copywriter Agent", "✍️")

    # Step 1: Load local image and encode as base64
    with open(image_path, "rb") as f:
        img_bytes = f.read()

    b64_img = base64.b64encode(img_bytes).decode("utf-8")

    # Step 2: Build OpenAI-compliant multimodal message
    messages = [
        {
            "role": "system",
            "content": "You are a copywriter that creates elegant campaign quotes based on an image and a marketing trend summary."
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{b64_img}",
                        "detail": "auto"
                    }
                },
                {
                    "type": "text",
                    "text": f"""
Here is a visual marketing image and a trend analysis:

Trend summary:
\"\"\"{trend_summary}\"\"\"

Please return a JSON object like:
{{
  "quote": "A short, elegant campaign phrase (max 12 words)",
  "justification": "Why this quote matches the image and trend"
}}"""
                }
            ]
        }
    ]

    # Step 3: Send request via aisuite
    response = client.chat.completions.create(
        model=model,
        messages=messages,
    )

    # Step 4: Parse JSON response
    content = response.choices[0].message.content.strip()

    utils.log_final_summary_html(content)

    try:
        match = re.search(r'\{.*\}', content, re.DOTALL)
        parsed = json.loads(match.group(0)) if match else {"error": "No valid JSON returned"}
    except Exception as e:
        parsed = {"error": f"Failed to parse: {e}", "raw": content}


    parsed["image_path"] = image_path
    return parsed


In [10]:
copywriter_agent_result = copywriter_agent(
    image_path=graphic_designer_agent_result["image_path"],
    trend_summary=market_research_result,
)


### 4.4 Packaging Agent


This agent packages everything into an executive-ready markdown report:

- Refines the trend summary for a CEO audience
- Includes the campaign visual
- Includes the campaign quote
- Includes the justification
- Writes to a `.md` file you can render in the notebook


In [12]:
def packaging_agent(trend_summary: str, image_url: str, quote: str, justification: str, output_path: str = "campaign_summary.md") -> str:

    """
    Packages the campaign assets into a beautifully formatted markdown report for executive review.

    Args:
        trend_summary (str): Summary of the market trends.
        image_url (str): URL of the campaign image.
        quote (str): Marketing quote to overlay.
        justification (str): Explanation for the quote.
        output_path (str): Path to save the markdown report.

    Returns:
        str: Path to the saved markdown file.
    """

    utils.log_agent_title_html("Packaging Agent", "📦")

    # We use this path in the src of the <img>
    styled_image_html = f"""
![Open the generated file to see]({image_url})
    """

    beautified_summary = client.chat.completions.create(
        model="openai:o4-mini",
        messages=[
            {"role": "system", "content": "You are a marketing communication expert writing elegant campaign summaries for executives."},
            {"role": "user", "content": f"""
Please rewrite the following trend summary to be clear, professional, and engaging for a CEO audience:

\"\"\"{trend_summary.strip()}\"\"\"
"""}
        ]
    ).choices[0].message.content.strip()

    utils.log_tool_result_html(beautified_summary)

    # Combine all parts into markdown
    markdown_content = f"""# 🕶️ Summer Sunglasses Campaign – Executive Summary

## 📊 Refined Trend Insights
{beautified_summary}

## 🎯 Campaign Visual
{styled_image_html}

## ✍️ Campaign Quote
{quote.strip()}

## ✅ Why This Works
{justification.strip()}

---

*Report generated on {datetime.now().strftime('%Y-%m-%d')}*
"""

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(markdown_content)

    return output_path



In [13]:
packaging_agent_result = packaging_agent(
    trend_summary=market_research_result,
    image_url=graphic_designer_agent_result["image_path"],
    quote=copywriter_agent_result.get("quote", ""),
    justification=copywriter_agent_result.get("justification", ""),
    output_path=f"campaign_summary_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.md",
)


In [14]:
# Load and render the markdown content
with open(packaging_agent_result, "r", encoding="utf-8") as f:
    md_content = f.read()
display(Markdown(md_content))


# 🕶️ Summer Sunglasses Campaign – Executive Summary

## 📊 Refined Trend Insights
Summer 2026 Sunglasses Trend Summary

As we prepare for a season defined by performance, glamour and nostalgia, we recommend spotlighting three distinct styles that align with consumer demand and reinforce our brand’s versatility.

1. Performance-Driven Wraparounds (SG004 “Sport”)  
   – Insight: One-piece shield lenses with sleek, continuous curves capture the modern athletic spirit.  
   – Why It Works: Ultra-light frames, flexible hinges and rubberized temple grips ensure both high-performance function and all-day comfort—ideal for adventure-seeking customers.

2. Elevated Cat-Eye Frames (SG003 “Mystique”)  
   – Insight: Sharp, angular corners and ornate temple detailing reinvent the classic cat-eye into a statement of contemporary femininity.  
   – Why It Works: Subtle upward sweeps at each brow line and polished accents mirror the latest runway looks, positioning “Mystique” as the go-to choice for fashion-savvy shoppers.

3. Retro-Chic Wayfarers (SG002 “Wayfarer”)  
   – Insight: Thick acetate frames and bold angles channel ’70s/’80s nostalgia with a Y2K twist—perfect for self-expressive styling.  
   – Why It Works: A confident silhouette that transitions effortlessly from beach to boulevard, catering to a broad audience drawn to vintage-inspired designs.

Recommendation  
Lead the Summer 2026 launch with SG004 “Sport” and SG003 “Mystique” as flagship “edgy” and “glam” offerings, then broaden your appeal by adding SG002 “Wayfarer.” This triad captures the season’s core themes—performance, elevated femininity and retro flair—ensuring comprehensive market coverage and maximum consumer engagement.

## 🎯 Campaign Visual

![Open the generated file to see](img-sF6u2Hmny3NoFlV1g7nIIvK8.png)
    

## ✍️ Campaign Quote
Golden Horizons: Sport Wraparound Meets Cat-Eye Couture

## ✅ Why This Works
This line captures the image’s sun-kissed beach glow and dynamic energy, while spotlighting the Summer 2026 trends—ultra-modern Sport shields and bold cat-eye silhouettes—into one elegant, evocative phrase.

---

*Report generated on 2026-02-19*


## 5. Full Campaign Pipeline – run_sunglasses_campaign_pipeline


In [15]:
def run_sunglasses_campaign_pipeline(output_path: str = "campaign_summary.md") -> dict:
    """
    Runs the full summer sunglasses campaign pipeline:
    1) Market research (search trends + match products)
    2) Generate visual + caption
    3) Generate quote based on image + trend
    4) Create executive markdown report

    Returns:
        dict: all intermediate results + path to final report
    """
    trend_summary = market_research_agent()
    print("✅ Market research completed")

    visual_result = graphic_designer_agent(trend_insights=trend_summary)
    image_path = visual_result["image_path"]
    print("🖼️ Image generated")

    quote_result = copywriter_agent(image_path=image_path, trend_summary=trend_summary)
    print("💬 Quote created")

    md_path = packaging_agent(
        trend_summary=trend_summary,
        image_url=image_path,
        quote=quote_result.get("quote", ""),
        justification=quote_result.get("justification", ""),
        output_path=f"campaign_summary_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.md",
    )
    print(f"📦 Report generated: {md_path}")

    return {
        "trend_summary": trend_summary,
        "visual": visual_result,
        "quote": quote_result,
        "markdown_path": md_path,
    }


In [16]:
results = run_sunglasses_campaign_pipeline()


✅ Market research completed


🖼️ Image generated


💬 Quote created


📦 Report generated: campaign_summary_2026-02-19_01-41-02.md


### 5.1 Results


In [17]:
with open(results["markdown_path"], "r", encoding="utf-8") as f:
    md_content = f.read()
display(Markdown(md_content))


# 🕶️ Summer Sunglasses Campaign – Executive Summary

## 📊 Refined Trend Insights
Summer 2026 Sunglasses Trends: Executive Summary

Overview  
As consumers seek both performance and personality, three distinct eyewear movements will define summer 2026. By combining sport-inspired innovation, refined metallic accents, and bold retro silhouettes, we can capture diverse lifestyle moments and drive premium growth.

1. Futuristic Shield & Wraparound Designs  
   – Single-lens, curved frames that marry athletic functionality with a forward-looking aesthetic.  
   – Appeal: Full-coverage protection and dynamic styling for active, trend-driven consumers.

2. Polished Metal Minimalism  
   – Sleek aviator shapes and ultra-thin metal details delivering a modern, high-end finish.  
   – Appeal: Timeless elegance with superior UV performance—ideal for both casual and upscale occasions.

3. Statement Cat-Eye & Vintage Shapes  
   – Dramatic upswept corners and heritage-inspired curves for expressive, fashion-forward looks.  
   – Appeal: A powerful vehicle for self-expression, tapping into the resurgence of retro glamour.

Strategic Catalog Recommendations  
Our top three SKUs directly align with these trends, ensuring a balanced summer assortment and maximizing cross-segment appeal.

1. SG004 “Sport”  
   • Why It Works: Oversized, wraparound single-lens delivers the futuristic shield look with integrated rubber grips for active lifestyles.  
   • Positioning: A performance-driven hero SKU that reinforces our leadership in functional innovation.

2. SG001 “Aviator”  
   • Why It Works: Lightweight metal frame and classic teardrop silhouette meet the polished-metal trend, offering enduring style and reliable UV protection.  
   • Positioning: A versatile cornerstone style that bridges sport and sophistication, driving high sell-through across price tiers.

3. SG003 “Mystique”  
   • Why It Works: Feminine cat-eye profile with subtle upsweep captures the retro revival, delivering a premium, statement-making option.  
   • Positioning: A fashion-first offering that enhances our brand’s luxury credentials and caters to self-expressive shoppers.

By spotlighting these three distinct silhouettes—futuristic shields, refined metals, and bold retro shapes—we secure a comprehensive summer lineup that speaks to every key customer segment and strengthens our market leadership.

## 🎯 Campaign Visual

![Open the generated file to see](img-TkYJm1g3CH8COPyjSc6mY5CE.png)
    

## ✍️ Campaign Quote
Summer Redefined: Futuristic Shields, Polished Metal, Retro Cat-Eyes

## ✅ Why This Works
This line highlights the three core Summer ’26 trends—oversized shield silhouettes, sleek metal aviator frames, and bold cat-eye styles—while echoing the image’s sunlit beach scenes, reflective lenses, and vintage-meets-futuristic vibe.

---

*Report generated on 2026-02-19*


## 6. Key Takeaways


- Use multi-agent LLM pipelines to automate a creative workflow end-to-end.
- Combine reasoning, tool-calling, and external data to ground outputs in reality.
- Use multimodal models to generate copy based on both text and images.
- Keep execution transparent and debuggable with structured logging.
- Deliver executive-ready reports in Markdown that blend insights, visuals, and justifications.
